# Solutions - Feature Engineering et Transformation des données

In [1]:
# Importation des bibliothèques
import pandas as pd
import numpy as np
import seaborn as sns
import plotly
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Pour afficher plus de colonnes dans les DataFrames
pd.set_option('display.max_columns', 30)

In [ ]:
# Chargement du jeu de données
df = pd.read_csv('../../data/passenger_satisfaction/train_50.csv')

# Copie du DataFrame pour ne pas modifier l'original
df_clean = df.copy()

# Pour les variables numériques, remplacer les valeurs manquantes par la médiane
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Pour les variables catégorielles, remplacer les valeurs manquantes par le mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

# Vérification que toutes les valeurs manquantes ont été traitées
print("Nombre total de valeurs manquantes après traitement :", df_clean.isnull().sum().sum())

## Exercice 1: Création de nouvelles features

Créez au moins deux nouvelles features qui pourraient être utiles pour prédire la satisfaction des passagers. Justifiez votre choix et analysez leur relation avec la variable cible.

In [ ]:
df_ex1 = df_clean.copy()

# Feature 1: Ratio entre la distance du vol et le retard total
# Justification: Un petit retard sur un vol court peut être plus frustrant qu'un retard similaire sur un vol long
df_ex1['Delay_Distance_Ratio'] = (df_ex1['Departure Delay in Minutes'] + df_ex1['Arrival Delay in Minutes']) / (df_ex1['Flight Distance'] + 1)  # +1 pour éviter division par zéro

# Feature 2: Score de confort global (combinaison de plusieurs critères liés au confort)
# Justification: Le confort global peut être un facteur déterminant de la satisfaction
comfort_cols = ['Seat comfort', 'Leg room service', 'Cleanliness', 'Food and drink', 'Inflight entertainment']
df_ex1['Comfort_Score'] = df_ex1[comfort_cols].mean(axis=1)

# Création des box plots avec plotly
fig = make_subplots(rows=1, cols=2, subplot_titles=('Ratio Retard/Distance vs Satisfaction', 
                                                   'Score de confort vs Satisfaction'))

# Delay_Distance_Ratio vs Satisfaction
fig.add_trace(
    go.Box(x=df_ex1['Satisfaction'], y=df_ex1['Delay_Distance_Ratio'], name='Delay/Distance'),
    row=1, col=1
)

# Comfort_Score vs Satisfaction
fig.add_trace(
    go.Box(x=df_ex1['Satisfaction'], y=df_ex1['Comfort_Score'], name='Comfort'),
    row=1, col=2
)

# Mise à jour du layout
fig.update_layout(
    title_text='Relation entre les nouvelles features et la satisfaction',
    showlegend=False,
    height=600,
    width=1200
)

# Échelle logarithmique pour le premier graphique
fig.update_yaxes(type="log", row=1, col=1)

# Affichage du graphique
fig.show()

## Exercice 2: Comparaison des méthodes d'encodage

Comparez l'impact de différentes méthodes d'encodage (Label Encoding vs One-Hot Encoding) sur une variable catégorielle de votre choix. Discutez des avantages et inconvénients de chaque méthode.

In [ ]:
# Exemple avec la variable 'Class'
df_ex2 = df_clean.copy()

# Label Encoding
le = LabelEncoder()
df_ex2['Class_Label_Encoded'] = le.fit_transform(df_ex2['Class'])
print("Label Encoding pour 'Class':")
for original, encoded in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {original} -> {encoded}")

# One-Hot Encoding
one_hot = pd.get_dummies(df_ex2['Class'], prefix='Class')
df_ex2 = pd.concat([df_ex2, one_hot], axis=1)
print("\nOne-Hot Encoding pour 'Class':")
print(one_hot.head())

# Visualisation avec plotly
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('Label Encoding', 'One-Hot Encoding'),
                    specs=[[{"type": "bar"}, {"type": "bar"}]])

# Distribution de la variable avec Label Encoding
label_counts = df_ex2['Class_Label_Encoded'].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=label_counts.index, y=label_counts.values, name='Label Encoding'),
    row=1, col=1
)

# Distribution de la variable avec One-Hot Encoding
fig.add_trace(
    go.Bar(x=one_hot.columns, y=one_hot.sum(), name='One-Hot Encoding'),
    row=1, col=2
)

# Mise à jour du layout
fig.update_layout(
    title_text='Comparaison des méthodes d\'encodage pour la variable Class',
    showlegend=False,
    height=600,
    width=1200
)

# Mise à jour des axes
fig.update_xaxes(title_text='Class (encodée)', row=1, col=1)
fig.update_xaxes(title_text='Class (one-hot)', row=1, col=2)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=2)

# Affichage du graphique
fig.show()

# Discussion
print("\nDiscussion:")
print("Label Encoding:")
print("  Avantages: Simple, conserve une seule colonne, peut capturer l'ordre si la variable est ordinale")
print("  Inconvénients: Introduit une relation d'ordre artificielle pour les variables nominales")
print("\nOne-Hot Encoding:")
print("  Avantages: Pas de relation d'ordre artificielle, meilleur pour les variables nominales")
print("  Inconvénients: Augmente le nombre de colonnes, peut créer de la multicolinéarité")

## Exercice 3: Création d'un pipeline personnalisé

Créez un pipeline personnalisé qui combine plusieurs étapes de prétraitement, y compris la création de nouvelles features, l'encodage des variables catégorielles et la standardisation des variables numériques.

In [ ]:
# Votre code ici
from sklearn.base import BaseEstimator, TransformerMixin

# Création d'un transformateur personnalisé pour la création de features
class FeatureCreator(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        
        # Calcul du retard total
        X_copy['Total_Delay'] = X_copy['Departure Delay in Minutes'] + X_copy['Arrival Delay in Minutes']
        
        # Score de confort
        comfort_cols = ['Seat comfort', 'Leg room service', 'Cleanliness', 'Food and drink', 'Inflight entertainment']
        X_copy['Comfort_Score'] = X_copy[comfort_cols].mean(axis=1)
        
        # Score de service
        service_cols = ['Inflight wifi service', 'On-board service', 'Baggage handling', 'Checkin service', 'Inflight service']
        X_copy['Service_Score'] = X_copy[service_cols].mean(axis=1)
        
        return X_copy

# Création du pipeline personnalisé
from sklearn.pipeline import Pipeline

# Définition des colonnes
categorical_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
numerical_cols = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
service_cols = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness'
]

# Colonnes à conserver pour la création de features
cols_to_keep = numerical_cols + categorical_cols + service_cols

# Pipeline complet
custom_pipeline = Pipeline([
    ('feature_creator', FeatureCreator()),
    ('preprocessor', ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), numerical_cols + ['Total_Delay', 'Comfort_Score', 'Service_Score']),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]), categorical_cols),
            ('serv', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), service_cols)
        ],
        remainder='drop'
    ))
])

# Test du pipeline
X = df.drop(['ID', 'Satisfaction'], axis=1)  # Features
y = df['Satisfaction']  # Variable cible

# Application du pipeline
X_transformed = custom_pipeline.fit_transform(X)
print(f"Forme des données transformées avec le pipeline personnalisé: {X_transformed.shape}")